In [1]:
%cd ../..

/run/media/nazif/2F946E411BA61D49/mirscribe


In [2]:
import pandas as pd 
import os
import gc

from scripts.utils import *
from scripts.features import generate_is_mirna_column
from scripts.ensembl import generate_transcript_id_and_gene_name_columns

In [3]:
def pipeline(file_name):
    
    df = pd.read_csv(f"data/sample_vcfs/{file_name}.vcf", sep="\t", header=None, names=["chr", "pos", "id", "ref", "alt"])
    # Add an 'id' column by combining 'chr', 'pos', 'ref', and 'alt' columns
    df['id'] = df["id"] + df['chr'] + '_' + df['pos'].astype(str) + '_' + df['ref'] + '_' + df['alt']

    # preparing mutation len values
    df.loc[:, "ref_len"] = df["ref"].str.len()
    df.loc[:, "alt_len"] = df["alt"].str.len()

    # if affected sequences are longer than 1, fetch nucleotides at the specified interval
    df.loc[df['ref_len'] > 1, 'fetched_nucleotides'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['pos'], x["pos"]+x["ref_len"]-1), axis=1)
    # if affected sequences are equal to 1, fetch nucleotides at the specified pos
    df.loc[df['ref_len'] == 1, 'fetched_nucleotides'] = df.apply(lambda x: get_nucleotide_at_position(x['chr'], x['pos']), axis=1)
    # same but for 0 length
    df.loc[df['ref_len'] == 0, 'fetched_nucleotides'] = ""

    # Add an 'is_nucleotides_same' column to check if fetched nucleotides are the same as 'ref'
    df.loc[:, 'is_nucleotides_same'] = df["fetched_nucleotides"] == df["ref"]
    # preparing reference indices
    df.loc[:, "ref_start"] = df["pos"]
    df.loc[:, "ref_end"] = df["pos"] + df["ref_len"]
    # preparing extended sequences
    df.loc[:, 'upstream_sequence'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['ref_start']-30, x["ref_start"]-1), axis=1)
    df.loc[:, 'downstream_sequence'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['ref_end'], x["ref_end"]+29), axis=1)
    # preparing wt sequence
    df.loc[:,'sequence'] = df["upstream_sequence"] + df["ref"] + df["downstream_sequence"]

    # preparing mutated sequence
    df.loc[:,'mutated_sequence'] = df["upstream_sequence"] + df["alt"] + df["downstream_sequence"]
    df = generate_is_mirna_column(df, grch=37)
    df = generate_transcript_id_and_gene_name_columns(df, grch=37)
    case_1 = df[df.is_mirna == 0]
    case_2 = df[df.is_mirna == 1]

    relevant_cols = ["id", "sequence", "mutated_sequence"]
    case_1 = case_1[relevant_cols]
    case_2 = case_2[relevant_cols]

    case_1.to_csv(f"results/{file_name}_case_1.csv", index=False)

    if len(case_2) > 0:
        print(f"{len(case_2)} case 2 mutations were found on {file_name}.")
        case_2.to_csv(f"results/{file_name}_case_2.csv", index=False)

    else:
        print(f"No case 2 mutations were found on {file_name}.")
        
    del df, case_1, case_2
    gc.collect()


In [4]:
import os
import pandas as pd

folder_path = "data/sample_vcfs"

# Get a list of all CSV files in the folder
csv_files = [os.path.splitext(file)[0] for file in os.listdir(folder_path) if file.endswith(".vcf")]

for file in csv_files:
    pipeline(file)

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


No case 2 mutations were found on PD3851a.


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


No case 2 mutations were found on PD3890a.


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


...

No case 2 mutations were found on PD4198a.


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


No case 2 mutations were found on PD4199a.


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /run/media/nazif/2F946E411BA61D49/data/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


No case 2 mutations were found on PD4248a.


In [ ]:
# Add an 'id' column by combining 'chr', 'pos', 'ref', and 'alt' columns
df['id'] = df["id"] + df['chr'] + '_' + df['pos'].astype(str) + '_' + df['ref'] + '_' + df['alt']

# preparing mutation len values
df.loc[:, "ref_len"] = df["ref"].str.len()
df.loc[:, "alt_len"] = df["alt"].str.len()

# if affected sequences are longer than 1, fetch nucleotides at the specified interval
df.loc[df['ref_len'] > 1, 'fetched_nucleotides'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['pos'], x["pos"]+x["ref_len"]-1), axis=1)
# if affected sequences are equal to 1, fetch nucleotides at the specified pos
df.loc[df['ref_len'] == 1, 'fetched_nucleotides'] = df.apply(lambda x: get_nucleotide_at_position(x['chr'], x['pos']), axis=1)
# same but for 0 length
df.loc[df['ref_len'] == 0, 'fetched_nucleotides'] = ""

# Add an 'is_nucleotides_same' column to check if fetched nucleotides are the same as 'ref'
df.loc[:, 'is_nucleotides_same'] = df["fetched_nucleotides"] == df["ref"]
df.is_nucleotides_same.value_counts()

In [ ]:
# preparing reference indices
df.loc[:, "ref_start"] = df["pos"]
df.loc[:, "ref_end"] = df["pos"] + df["ref_len"]


In [ ]:
# preparing extended sequences
df.loc[:, 'upstream_sequence'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['ref_start']-30, x["ref_start"]-1), axis=1)
df.loc[:, 'downstream_sequence'] = df.apply(lambda x: get_nucleotides_in_interval(x['chr'], x['ref_end'], x["ref_end"]+29), axis=1)

In [ ]:
# preparing wt sequence
df.loc[:,'sequence'] = df["upstream_sequence"] + df["ref"] + df["downstream_sequence"]

# preparing mutated sequence
df.loc[:,'mutated_sequence'] = df["upstream_sequence"] + df["alt"] + df["downstream_sequence"]

In [ ]:
df = generate_is_mirna_column(df, grch=37)
df = generate_transcript_id_and_gene_name_columns(df, grch=37)

In [ ]:
if df.is_nucleotides_same.all():
    print("ha")

In [ ]:
case_1 = df[df.is_mirna == 0]
case_2 = df[df.is_mirna == 1]

relevant_cols = ["id", "sequence", "mutated_sequence"]
case_1 = case_1[relevant_cols]
case_2 = case_2[relevant_cols]

case_1.to_csv(f"results/{file_name}_case_1.csv", index=False)

if len(case_2) > 0:
    print(f"{len(case_2)} mutations were found on {file_name}.")
    case_2.to_csv(f"results/{file_name}_case_2.csv", index=False)

else:
    print(f"No case 2 mutations were found on {file_name}.")